# Raw input viewer for PyPSA-Earth Taiwan

This notebook inspects the Taiwan workflow inputs and intermediate PyPSA elements before results analysis. It is organized around the basic PyPSA objects: buses, lines, transformers, generators, loads, storage units, links/stores, and renewable potential profiles.

Default run: `tw_test1_highs_2013_7d_4h_6b`. Helper functions live in `viewer_helper.py`.


In [59]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import pypsa
import xarray as xr

cwd = Path.cwd().resolve()
helper_candidates = [cwd, cwd / "pypsa_tw" / "viewer", cwd / "viewer"]
helper_candidates += [parent / "pypsa_tw" / "viewer" for parent in cwd.parents]
helper_candidates += [parent / "viewer" for parent in cwd.parents]
for candidate in helper_candidates:
    if (candidate / "viewer_helper.py").exists():
        sys.path.insert(0, str(candidate))
        break

from viewer_helper import (
    component_capacity,
    isolated_buses,
    network_graph,
    read_table,
    resolve_repo,
    show_head,
    table_info,
    topology_summary,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 180)

REPO = resolve_repo()

# RUN = "tw_test1_highs_2013_7d_4h_6b"
RUN = "tw_test1_gurobi_2013_7d_4h_6b"

RESOURCES = REPO / "resources" / RUN
NETWORKS = REPO / "networks" / RUN
RESULTS = REPO / "results" / RUN

print("repo:", REPO)
print("run:", RUN)
print("resources exists:", RESOURCES.exists())
print("networks exists:", NETWORKS.exists())
print("results exists:", RESULTS.exists())


repo: f:\Barton\Repositories\pypsa-earth
run: tw_test1_gurobi_2013_7d_4h_6b
resources exists: True
networks exists: True
results exists: True


## Raw OSM and cleaned OSM tables

These files are the raw spatial/electrical source tables before they become a PyPSA network.

In [60]:
osm_paths = {
    "raw_substations": RESOURCES / "osm" / "raw" / "all_raw_substations.csv",
    "raw_lines": RESOURCES / "osm" / "raw" / "all_raw_lines.csv",
    "raw_generators": RESOURCES / "osm" / "raw" / "all_raw_generators.csv",
    "raw_cables": RESOURCES / "osm" / "raw" / "all_raw_cables.csv",
    "clean_substations": RESOURCES / "osm" / "clean" / "all_clean_substations.geojson",
    "clean_lines": RESOURCES / "osm" / "clean" / "all_clean_lines.geojson",
    "clean_generators": RESOURCES / "osm" / "clean" / "all_clean_generators.csv",
}

osm_tables = {name: read_table(path) for name, path in osm_paths.items()}
pd.DataFrame([table_info(name, df) for name, df in osm_tables.items()])

,table,rows,columns,column_names
0,raw_substations,734,25,"id, lonlat, Type, Region, refs, tags.operator,..."
1,raw_lines,1876,17,"id, lonlat, Type, Region, refs, tags.cables, t..."
2,raw_generators,1775,17,"id, lonlat, Type, Region, refs, tags.generator..."
3,raw_cables,253,20,"id, lonlat, Type, Region, refs, tags.circuits,..."
4,clean_substations,3183,12,"bus_id, symbol, tag_substation, voltage, lon, ..."
5,clean_lines,1531,13,"line_id, tag_type, voltage, circuits, bus0, bu..."
6,clean_generators,141,18,"Unnamed: 0, id, Type, Region, tags.generator:m..."


In [61]:
show_head(osm_tables["raw_substations"], 8)

,id,lonlat,Type,Region,refs,tags.operator,tags.operator:en,tags.operator:wikidata,tags.operator:zh,tags.power,tags.barrier,tags.name,tags.substation,tags.location,tags.voltage,tags.building,tags.alt_name,tags.alt_name:en,tags.alt_name:zh,tags.name:en,tags.name:zh,tags.rating,tags.utility,tags.landuse,other_tags
0,33406778,"[(120.30996380000025, 22.649209099999798), (12...",area,TW,"[380043487, 7470226402, 380043488, 380043489, ...",台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,substation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,110413202,"[(121.23963300000044, 24.972486699999994), (12...",area,TW,"[1260939964, 1260939962, 1260939984, 506901065...",NaN,NaN,NaN,NaN,substation,wall,自立變電所,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,110413203,"[(121.22296830000045, 24.966737), (121.2233372...",area,TW,"[1260939986, 1260939971, 1260939967, 126093997...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,110413204,"[(121.24325000000044, 24.990103199999997), (12...",area,TW,"[1260939969, 1260939960, 5027685274, 126093996...",NaN,NaN,NaN,NaN,substation,wall,中壢一次變電所,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,110413205,"[(121.26649940000044, 24.9753801), (121.266638...",area,TW,"[1260939980, 1260939982, 1260939951, 126093997...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,112022553,"[(121.21401770000006, 25.02115729999996), (121...",area,TW,"[1274547644, 1274547642, 1274547643, 127454764...",NaN,NaN,NaN,NaN,substation,NaN,五權變電所,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,114397074,"[(121.23063790000022, 24.944885000000042), (12...",area,TW,"[1296013091, 1296013093, 1296013100, 129601308...",台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,substation,wall,變電所,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,114665430,"[(121.28620020000048, 24.913371600000094), (12...",area,TW,"[1298320182, 5027458928, 1298320184, 129832018...",NaN,NaN,NaN,NaN,substation,wall,松樹一次變電所,transmission,outdoor,161000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'tags.frequency': '60'}


In [62]:
show_head(osm_tables["raw_lines"], 8)

,id,lonlat,Type,Region,refs,tags.cables,tags.name,tags.operator,tags.operator:en,tags.operator:wikidata,tags.operator:zh,tags.power,tags.voltage,tags.wires,tags.circuits,tags.line,other_tags
0,52174646,"[(121.48747829999998, 25.029060299999983), (12...",way,TW,"[664646856, 664646859, 664646865, 664646874, 6...",6,板橋～城中線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,161000,double,NaN,NaN,"{'tags.alt_name': '板橋～城中～成都線', 'tags.layer': '5'}"
1,52174647,"[(121.48747829999998, 25.029060299999983), (12...",way,TW,"[664646856, 664646949]",9,板橋～萬華線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,69000,double;single,NaN,NaN,"{'tags.alt_name': '埔墘～萬華線', 'tags.layer': '5'}"
2,52176335,"[(121.49858719999996, 25.00788949999999), (121...",way,TW,"[664682388, 664682393, 664682396, 664682400, 6...",6,板橋～青年線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,161000,double,NaN,NaN,{'tags.alt_name': '板橋～南海線'}
3,52176340,"[(121.53137789999998, 25.010799199999994), (12...",way,TW,"[664682459, 664682460, 664682461, 664682463]",6,台北～古亭線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,69000,single,NaN,NaN,"{'tags.alt_name': '台北～水源線', 'tags.layer': '5'}"
4,52176341,"[(121.53495649999998, 25.00269059999999), (121...",way,TW,"[664682463, 664682477, 4957721421]",6,台北～永和線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,69000,single,NaN,NaN,NaN
5,52176345,"[(121.53137789999998, 25.010799199999994), (12...",way,TW,"[664682459, 664682490, 4948423516, 664682495]",3,台北～古亭線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,69000,single,NaN,NaN,{'tags.layer': '5'}
6,52187035,"[(121.55395999999996, 25.013878900000005), (12...",way,TW,"[664706953, 664706962, 664706967, 664706972, 6...",6,深美～台北～臥龍線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,161000,double,NaN,NaN,"{'tags.alt_name': '深美～台北～建國線', 'tags.name:zh':..."
7,52187094,"[(121.62261699999995, 25.00355410000005), (121...",way,TW,"[1379652367, 1489564811, 1489421564, 148942159...",6,深美～台北線,台灣電力公司,Taiwan Power Company,Q711691,台灣電力公司,line,161000,double,NaN,NaN,{'tags.source': 'Yahoo'}


In [63]:
voltage_rows = []
for name, df in osm_tables.items():
    for col in ["tags.voltage", "voltage"]:
        if col in df.columns:
            counts = df[col].value_counts(dropna=False).head(20)
            voltage_rows.extend(
                {"table": name, "column": col, "value": value, "count": count}
                for value, count in counts.items()
            )
pd.DataFrame(voltage_rows)

,table,column,value,count
0,raw_substations,tags.voltage,NaN,339
1,raw_substations,tags.voltage,69000,127
2,raw_substations,tags.voltage,161000,109
3,raw_substations,tags.voltage,69000;22000,36
4,raw_substations,tags.voltage,161000;22000,35
5,raw_substations,tags.voltage,161000;69000,31
6,raw_substations,tags.voltage,345000;161000,21
7,raw_substations,tags.voltage,161000;25000,9
8,raw_substations,tags.voltage,22800;220;120,9
9,raw_substations,tags.voltage,345000;161000;69000,4


## Built base-network CSVs

These are the cleaned and spatially merged tables that feed `base_network.py`.

In [64]:
base_paths = {
    "buses": RESOURCES / "base_network" / "all_buses_build_network.csv",
    "lines": RESOURCES / "base_network" / "all_lines_build_network.csv",
    "transformers": RESOURCES / "base_network" / "all_transformers_build_network.csv",
    "converters": RESOURCES / "base_network" / "all_converters_build_network.csv",
    "powerplants": RESOURCES / "powerplants.csv",
    "powerplants_osm2pm": RESOURCES / "powerplants_osm2pm.csv",
    "demand_profiles": RESOURCES / "demand_profiles.csv",
    "costs_2030_elec": RESOURCES / "costs_2030_elec.csv",
}

base_tables = {name: read_table(path) for name, path in base_paths.items()}
pd.DataFrame([table_info(name, df) for name, df in base_tables.items()])

,table,rows,columns,column_names
0,buses,193,14,"Unnamed: 0, bus_id, station_id, voltage, dc, s..."
1,lines,243,10,"Unnamed: 0, line_id, circuits, voltage, bus0, ..."
2,transformers,69,5,"Unnamed: 0, line_id, bus0, bus1, geometry"
3,converters,0,5,"Unnamed: 0, converter_id, bus0, bus1, geometry"
4,powerplants,118,20,"Unnamed: 0, Name, Fueltype, Technology, Set, C..."
5,powerplants_osm2pm,0,0,
6,demand_profiles,168,125,"time, 0, 1, 4, 6, 9, 12, 14, 17, 20, 23, 25 ..."
7,costs_2030_elec,297,61,"technology, Bottom storage temperature, C in f..."


In [65]:
show_head(base_tables["buses"], 12)

,Unnamed: 0,bus_id,station_id,voltage,dc,symbol,under_construction,tag_substation,tag_area,lon,lat,country,geometry,substation_lv
0,0,0,0,161000,False,substation,False,transmission,0.0,121.2840,24.9193,TW,POINT (121.284 24.9193),True
1,1,1,1,69000,False,substation,False,transmission,0.0,120.6100,24.1915,TW,POINT (120.61 24.1915),True
2,2,2,1,161000,False,substation,False,transmission,0.0,120.6110,24.1925,TW,POINT (120.611 24.1925),False
3,3,3,1,345000,False,substation,False,transmission,0.0,120.6120,24.1935,TW,POINT (120.612 24.1935),False
4,4,4,2,161000,False,substation,False,transmission,0.0,121.1943,24.8187,TW,POINT (121.1943 24.8187),True
5,5,5,2,345000,False,substation,False,transmission,0.0,121.1953,24.8197,TW,POINT (121.1953 24.8197),False
6,6,6,3,69000,False,substation,False,transmission,0.0,121.5113,25.0216,TW,POINT (121.5113 25.0216),True
7,7,7,3,161000,False,substation,False,transmission,0.0,121.5123,25.0226,TW,POINT (121.5123 25.0226),False
8,8,8,3,345000,False,substation,False,transmission,0.0,121.5133,25.0236,TW,POINT (121.5133 25.0236),False
9,9,9,4,69000,False,substation,False,transmission,0.0,121.7290,24.6513,TW,POINT (121.729 24.6513),True


In [66]:
show_head(base_tables["lines"], 12)

,Unnamed: 0,line_id,circuits,voltage,bus0,bus1,length,dc,geometry,under_construction
0,0,52187094-1_0,2.0,161000,62,7,4165.769525,False,MULTILINESTRING ((121.622617 25.00355410000004...,False
1,1,103774600-1_2,2.0,345000,137,3,13855.273474,False,MULTILINESTRING ((120.63060739999987 24.086680...,False
2,2,103964830-1_0,2.0,161000,59,56,19417.904292,False,"LINESTRING (120.487 24.0605, 120.5177697000000...",False
3,3,105013657-1_0,2.0,345000,3,60,9620.220641,False,MULTILINESTRING ((120.4945969000001 24.1901868...,False
4,4,114553859-1_0,2.0,345000,74,82,45727.076342,False,"MULTILINESTRING ((121.2885 25.1127, 121.363850...",False
5,5,121981985-1_0,2.0,345000,60,3,10780.477913,False,"MULTILINESTRING ((120.488 24.0615, 120.5448176...",False
6,6,121997903-1_0,1.0,345000,79,80,11963.857667,False,MULTILINESTRING ((121.66088519999998 25.199748...,False
7,7,122063472-1_0,2.0,345000,80,83,15193.299780,False,MULTILINESTRING ((121.66079579999996 25.200150...,False
8,8,122066598-1_0,1.0,345000,81,80,5669.443533,False,"MULTILINESTRING ((121.6131 25.2044, 121.613069...",False
9,9,122190580-1_0,2.0,345000,60,3,4872.439766,False,"MULTILINESTRING ((120.488 24.0615, 120.5448176...",False


In [67]:
bus_df = base_tables["buses"]
line_df = base_tables["lines"]
pd.concat(
    {
        "base_buses_by_voltage": bus_df["voltage"].value_counts().sort_index() if "voltage" in bus_df else pd.Series(dtype=int),
        "base_lines_by_voltage": line_df["voltage"].value_counts().sort_index() if "voltage" in line_df else pd.Series(dtype=int),
    },
    axis=1,
)

,base_buses_by_voltage,base_lines_by_voltage
voltage,,
69000,75,74.0
161000,73,101.0
345000,44,68.0
690000,1,NaN


## PyPSA networks through the workflow

`base.nc`, `elec.nc`, `elec_s.nc`, and `elec_s_6.nc` show how raw data becomes the clustered electricity model.

In [68]:
network_paths = {
    "base": NETWORKS / "base.nc",
    "elec": NETWORKS / "elec.nc",
    "elec_s": NETWORKS / "elec_s.nc",
    "elec_s_6": NETWORKS / "elec_s_6.nc",
    "elec_s_6_ec": NETWORKS / "elec_s_6_ec.nc",
    "prepared": NETWORKS / "elec_s_6_ec_lcopt_Co2L-4H.nc",
}

networks = {name: pypsa.Network(path) for name, path in network_paths.items() if path.exists()}
pd.DataFrame(
    [
        {
            "network": name,
            "buses": len(n.buses),
            "lines": len(n.lines),
            "transformers": len(n.transformers) if hasattr(n, "transformers") else 0,
            "links": len(n.links),
            "generators": len(n.generators),
            "loads": len(n.loads),
            "storage_units": len(n.storage_units),
            "stores": len(n.stores),
            "snapshots": len(n.snapshots),
            "isolated_buses": ", ".join(isolated_buses(n, include_transformers=True)),
        }
        for name, n in networks.items()
    ]
)

INFO:pypsa.io:Imported network base.nc has buses, lines, transformers
INFO:pypsa.io:Imported network elec.nc has buses, carriers, generators, lines, loads, storage_units, transformers
INFO:pypsa.io:Imported network elec_s.nc has buses, carriers, generators, lines, loads, storage_units
INFO:pypsa.io:Imported network elec_s_6.nc has buses, carriers, generators, lines, loads, storage_units
INFO:pypsa.io:Imported network elec_s_6_ec.nc has buses, carriers, generators, lines, loads, storage_units
INFO:pypsa.io:Imported network elec_s_6_ec_lcopt_Co2L-4H.nc has buses, carriers, generators, global_constraints, lines, loads, storage_units


,network,buses,lines,transformers,links,generators,loads,storage_units,stores,snapshots,isolated_buses
0,base,193,243,69,0,0,0,0,0,168,"135, 77, 78"
1,elec,193,243,69,0,418,124,15,0,168,"135, 77, 78"
2,elec_s,79,130,0,0,276,84,15,0,168,
3,elec_s_6,6,7,0,0,38,6,5,0,168,
4,elec_s_6_ec,6,7,0,0,38,6,5,0,168,
5,prepared,6,7,0,0,38,6,5,0,42,


In [69]:
isolated_rows = []
for network_name, network in networks.items():
    isolated = isolated_buses(network, include_transformers=True)
    isolated_rows.append(
        {
            "network": network_name,
            "bus_count": len(network.buses),
            "isolated_bus_count": len(isolated),
            "isolated_buses": ", ".join(isolated),
        }
    )

pd.DataFrame(isolated_rows)


,network,bus_count,isolated_bus_count,isolated_buses
0,base,193,3,"135, 77, 78"
1,elec,193,3,"135, 77, 78"
2,elec_s,79,0,
3,elec_s_6,6,0,
4,elec_s_6_ec,6,0,
5,prepared,6,0,


In [70]:
for network_name, network in networks.items():
    buses_map = network.buses.reset_index(names="bus").copy()
    if "x" not in buses_map.columns and "lon" in buses_map.columns:
        buses_map["x"] = buses_map["lon"]
    if "y" not in buses_map.columns and "lat" in buses_map.columns:
        buses_map["y"] = buses_map["lat"]

    buses_map = buses_map.dropna(subset=["x", "y"])
    buses_map["is_isolated"] = buses_map["bus"].isin(
        isolated_buses(network, include_transformers=True)
    )
    buses_map["degree"] = (
        buses_map["bus"]
        .map(dict(network_graph(network).degree()))
        .fillna(0)
        .astype(int)
    )

    fig = px.scatter_mapbox(
        buses_map,
        lat="y",
        lon="x",
        text="bus",
        color="is_isolated",
        hover_data=["bus", "v_nom", "carrier", "degree"],
        zoom=6.2,
        height=650,
        title=f"Bus locations: {network_name}",
    )
    fig.update_traces(marker={"size": 10})
    fig.update_layout(
        mapbox_style="open-street-map",
        margin=dict(l=0, r=0, t=50, b=0),
    )
    fig.show()


C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3014535771.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/py

C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3014535771.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/py

C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3014535771.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/py

C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3014535771.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/py

C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3014535771.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/py

C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3014535771.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/py

In [71]:
topology_summary(networks["elec_s_6"], include_transformers=True)

,component_index,bus_count,sample_buses
0,0,6,"TW0 0, TW0 1, TW0 2, TW0 3, TW0 4, TW0 5"


## Buses

In [72]:
n = networks["elec_s_6"]
graph = network_graph(n)
buses = n.buses.copy()
buses["degree"] = pd.Series(dict(graph.degree())).reindex(buses.index).fillna(0).astype(int)
buses["is_isolated"] = buses["degree"].eq(0)
buses.reset_index(names="bus")

,bus,v_nom,tag_substation,tag_area,lon,lat,country,x,y,control,generator,type,carrier,unit,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,sub_network,degree,is_isolated
0,TW0 0,380.0,transmission,0.0,121.701158,25.037000,TW,121.701158,25.037000,Slack,TW0 0 coal-2000,,AC,,1.0,0.0,inf,,1,False
1,TW0 1,380.0,transmission,0.0,120.286815,23.105610,TW,120.286815,23.105610,PQ,,,AC,,1.0,0.0,inf,,3,False
2,TW0 2,380.0,transmission,0.0,121.014531,24.055000,TW,121.014531,24.055000,PQ,,,AC,,1.0,0.0,inf,,4,False
3,TW0 3,380.0,transmission,0.0,121.166313,24.878637,TW,121.166313,24.878637,PQ,,,AC,,1.0,0.0,inf,,2,False
4,TW0 4,380.0,transmission,0.0,120.650560,22.450247,TW,120.650560,22.450247,PQ,,,AC,,1.0,0.0,inf,,2,False
5,TW0 5,380.0,transmission,0.0,120.453416,23.719263,TW,120.453416,23.719263,PQ,,,AC,,1.0,0.0,inf,,2,False


In [73]:
fig = px.scatter_mapbox(
    buses.reset_index(names="bus"),
    lat="y",
    lon="x",
    text="bus",
    color="is_isolated",
    hover_data=["v_nom", "country", "degree"],
    zoom=6.4,
    height=650,
    title="Clustered Taiwan buses: elec_s_6",
)
fig.update_layout(mapbox_style="open-street-map", margin=dict(l=0, r=0, t=45, b=0))
fig.show()

C:\Users\chyi\AppData\Local\Temp\6\ipykernel_19736\3408367142.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:1054: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace_type = constructor().type
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\plotly\express\_core.py:2559: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  subplot_type = _subplot_type_for_trace_type(constructor().type)
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/pyt

## Lines and transformers

In [74]:
line_cols = [c for c in ["bus0", "bus1", "carrier", "type", "v_nom", "length", "s_nom", "s_nom_min", "s_nom_max", "s_nom_extendable", "num_parallel"] if c in n.lines.columns]
n.lines[line_cols].reset_index(names="line")

,line,bus0,bus1,carrier,type,v_nom,length,s_nom,s_nom_min,s_nom_max,s_nom_extendable,num_parallel
0,0,TW0 0,TW0 3,AC,Al/St 240/40 4-bundle 380.0,380.0,70.901389,13926.675762,0.0,inf,False,8.201316
1,1,TW0 1,TW0 2,AC,Al/St 240/40 4-bundle 380.0,380.0,161.265510,3083.396848,0.0,inf,False,1.815789
2,2,TW0 1,TW0 4,AC,Al/St 240/40 4-bundle 380.0,380.0,102.325887,6243.878616,0.0,inf,False,3.676974
3,3,TW0 1,TW0 5,AC,Al/St 240/40 4-bundle 380.0,380.0,87.901047,7400.152434,0.0,inf,False,4.357895
4,4,TW0 2,TW0 3,AC,Al/St 240/40 4-bundle 380.0,380.0,116.079597,9969.649807,0.0,inf,False,5.871053
5,5,TW0 2,TW0 4,AC,Al/St 240/40 4-bundle 380.0,380.0,227.841510,179.864816,0.0,inf,False,0.105921
6,6,TW0 2,TW0 5,AC,Al/St 240/40 4-bundle 380.0,380.0,85.222626,26234.568179,0.0,inf,False,15.449342


In [75]:
elec = networks["elec"]
trafo_cols = [c for c in ["bus0", "bus1", "type", "model", "s_nom", "x", "r", "g", "b"] if c in elec.transformers.columns]
elec.transformers[trafo_cols].reset_index(names="transformer").head(80)

,transformer,bus0,bus1,type,model,s_nom,x,r,g,b
0,transf_1_0,1,2,,t,2000.0,0.1,0.0,0.0,0.0
1,transf_1_1,2,3,,t,2000.0,0.1,0.0,0.0,0.0
2,transf_2_0,4,5,,t,2000.0,0.1,0.0,0.0,0.0
3,transf_3_0,6,7,,t,2000.0,0.1,0.0,0.0,0.0
4,transf_3_1,7,8,,t,2000.0,0.1,0.0,0.0,0.0
5,transf_4_0,9,10,,t,2000.0,0.1,0.0,0.0,0.0
6,transf_4_1,10,11,,t,2000.0,0.1,0.0,0.0,0.0
7,transf_5_0,12,13,,t,2000.0,0.1,0.0,0.0,0.0
8,transf_6_0,14,15,,t,2000.0,0.1,0.0,0.0,0.0
9,transf_6_1,15,16,,t,2000.0,0.1,0.0,0.0,0.0


## Generators and power plants

In [76]:
gen = n.generators.copy()
gen["capacity_MW"] = component_capacity(gen)
gen_summary = (
    gen.groupby(["carrier", "p_nom_extendable"], dropna=False)["capacity_MW"]
    .agg(["count", "sum", "min", "max"])
    .reset_index()
    .sort_values("sum", ascending=False)
)
gen_summary

,carrier,p_nom_extendable,count,sum,min,max
0,CCGT,False,4,18317.264795,2118.000000,9431.264795
1,coal,False,6,18241.000000,458.000000,5632.000000
8,solar,False,6,12417.685000,763.176895,3186.872661
2,nuclear,False,2,5317.773580,1902.000000,3415.773580
3,offwind-ac,False,6,2216.000000,0.000000,1712.000000
6,onwind,False,6,1104.822000,51.150145,470.994935
7,ror,False,1,1000.000000,1000.000000,1000.000000
5,oil,False,1,277.900649,277.900649,277.900649
4,offwind-dc,False,6,0.000000,0.000000,0.000000


In [77]:
gen_cols = [c for c in ["bus", "carrier", "p_nom", "p_nom_min", "p_nom_max", "p_nom_extendable", "capital_cost", "marginal_cost", "build_year", "lifetime"] if c in gen.columns]
gen[gen_cols].reset_index(names="generator").sort_values(["bus", "carrier"]).head(120)

,generator,bus,carrier,p_nom,p_nom_min,p_nom_max,p_nom_extendable,capital_cost,marginal_cost,build_year,lifetime
0,TW0 0 coal,TW0 0,coal,1644.000000,1644.000000,1.644000e+03,False,337208.027448,30.098840,2001,45.000000
1,TW0 0 nuclear,TW0 0,nuclear,3415.773580,3415.773580,3.415774e+03,False,753784.215297,14.013271,1989,47.333333
2,TW0 0 offwind-ac,TW0 0,offwind-ac,0.000000,0.000000,8.142924e+02,False,202085.033127,0.015000,0,30.000000
3,TW0 0 offwind-dc,TW0 0,offwind-dc,0.000000,0.000000,3.369633e+01,False,213125.359908,0.015000,0,30.000000
4,TW0 0 onwind,TW0 0,onwind,53.842258,0.000000,8.182361e+03,False,101644.123324,0.015000,0,30.000000
5,TW0 0 solar,TW0 0,solar,1387.594355,0.000000,3.350597e+03,False,39296.472708,0.010000,0,40.000000
6,TW0 1 CCGT,TW0 1,CCGT,3510.000000,3510.000000,3.510000e+03,False,104788.020783,46.803121,1998,40.000000
7,TW0 1 coal,TW0 1,coal,2148.000000,2148.000000,2.148000e+03,False,337208.027448,30.098840,1991,44.000000
8,TW0 1 offwind-ac,TW0 1,offwind-ac,0.000000,0.000000,1.911690e+03,False,202885.214456,0.015000,0,30.000000
9,TW0 1 offwind-dc,TW0 1,offwind-dc,0.000000,0.000000,2.363064e+04,False,219092.939562,0.015000,0,30.000000


In [78]:
show_head(base_tables["powerplants"], 30)

,Unnamed: 0,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,DateIn,DateRetrofit,DateOut,lat,lon,EIC,projectID,bus
0,0,Datan,CCGT,CCGT,PP,TW,6102.000000,NaN,NaN,0.0,0.0,0.0,2006.0,2006.0,2022.0,25.029685,121.048220,"{nan, nan, nan, nan, nan, nan, nan, nan, nan}","{'GEM': {'L100000406335'}, 'GEO': {'GEO-2850'}...",184
1,1,Mingtan,Hydro,Pumped Storage,Store,TW,1602.000000,NaN,NaN,0.0,0.0,0.0,1993.0,1993.0,2093.0,23.836800,120.869500,{nan},"{'GEM': {'G603578'}, 'GEO': {'GEO-45427'}, 'GP...",67
2,2,Ho Ping,Hard Coal,Steam Turbine,PP,TW,1320.000000,NaN,NaN,0.0,0.0,0.0,2002.0,2002.0,2047.0,24.307782,121.763460,"{nan, nan}","{'GEM': {'G100000103903', 'G100000103902'}, 'G...",138
3,3,Star Buck,CCGT,CCGT,PP,TW,490.000000,NaN,NaN,0.0,0.0,0.0,2009.0,2009.0,2049.0,24.129192,120.422142,{nan},"{'GEM': {'L100000406344'}, 'GEO': {'GEO-5531'}...",58
4,4,Chia Hui,CCGT,CCGT,CHP,TW,1210.000000,NaN,NaN,0.0,0.0,0.0,2003.0,2003.0,2043.0,23.531781,120.472716,"{nan, nan}","{'GEM': {'L100000406346'}, 'GEO': {'GEO-2655'}...",45
5,5,Taichung,Hard Coal,Steam Turbine,PP,TW,5500.000000,NaN,NaN,0.0,0.0,0.0,1991.0,1991.0,2035.0,24.214365,120.482627,"{nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","{'GEM': {'G100000109826', 'G100000109825', 'G1...",1
6,6,Nanpu,CCGT,CCGT,PP,TW,1118.000000,NaN,NaN,0.0,0.0,0.0,1995.0,1995.0,2035.0,22.600761,120.299218,"{nan, nan, nan, nan}","{'GEM': {'L100000406338'}, 'GEO': {'GEO-4437'}...",39
7,7,Talin,CCGT,CCGT,PP,TW,1000.000000,NaN,NaN,0.0,0.0,0.0,1975.0,1975.0,2022.0,22.537083,120.328797,"{nan, nan}","{'GEM': {'L100000103542'}, 'GEO': {'GEO-5646'}...",30
8,8,Mailiao,Hard Coal,Steam Turbine,PP,TW,4500.000000,NaN,NaN,0.0,0.0,0.0,1997.0,1997.0,2025.0,23.805649,120.198551,"{nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","{'GEM': {'G100000106586', 'G100000106583', 'G1...",120
9,9,Maanshan,Nuclear,Steam Turbine,PP,TW,1902.000000,NaN,NaN,0.0,0.0,0.0,1984.0,1984.0,2034.0,21.958900,120.750100,"{nan, nan}","{'GEM': {'G100000500629', 'G100000500628'}, 'G...",99


## Loads and demand profile

In [79]:
loads = n.loads.copy()
load_profile = n.loads_t.p_set.copy()
load_summary = pd.DataFrame(
    {
        "load": load_profile.columns,
        "bus": loads.reindex(load_profile.columns)["bus"].values,
        "mean_MW": load_profile.mean().values,
        "max_MW": load_profile.max().values,
        "min_MW": load_profile.min().values,
    }
)
load_summary

,load,bus,mean_MW,max_MW,min_MW
0,TW0 0,TW0 0,12164.541968,14175.906295,9457.280244
1,TW0 1,TW0 1,3407.038164,3970.379969,2648.789803
2,TW0 2,TW0 2,8269.746144,9637.119649,6429.284969
3,TW0 3,TW0 3,2389.440890,2784.526556,1857.662391
4,TW0 4,TW0 4,4805.669256,5600.269804,3736.150612
5,TW0 5,TW0 5,4059.846170,4731.127488,3156.313085


In [80]:
total_load = load_profile.sum(axis=1)
fig = px.line(total_load.reset_index(), x="snapshot", y=0, labels={0: "MW"}, title="Total demand in selected PyPSA network")
fig.show()

## Storage units, links, and stores

In [81]:
storage = n.storage_units.copy()
if not storage.empty:
    storage["capacity_MW"] = component_capacity(storage)
storage.reset_index(names="storage_unit")

,storage_unit,carrier,bus,p_nom,capital_cost,max_hours,efficiency_store,efficiency_dispatch,cyclic_state_of_charge,build_year,lifetime,p_min_pu,control,type,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_max_pu,p_set,q_set,sign,spill_cost,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,state_of_charge_initial,state_of_charge_initial_per_period,state_of_charge_set,cyclic_state_of_charge_per_period,standing_loss,inflow,p_nom_opt,capacity_MW
0,TW0 0 hydro,hydro,TW0 0,90.0,0.000000,6.0,0.000000,0.900000,True,1964,100.0,0.0,,,0.0,False,0.0,inf,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False,NaN,True,0.0,0.0,0.0,90.0
1,TW0 2 PHS,PHS,TW0 2,2712.0,182698.734592,0.0,0.866025,0.866025,True,1963,100.0,-1.0,,,0.0,False,0.0,inf,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False,NaN,True,0.0,0.0,0.0,2712.0
2,TW0 2 hydro,hydro,TW0 2,1349.5,0.000000,6.0,0.000000,0.900000,True,1991,100.0,0.0,,,0.0,False,0.0,inf,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False,NaN,True,0.0,0.0,0.0,1349.5
3,TW0 3 hydro,hydro,TW0 3,18.0,0.000000,6.0,0.000000,0.900000,True,1941,100.0,0.0,,,0.0,False,0.0,inf,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False,NaN,True,0.0,0.0,0.0,18.0
4,TW0 4 hydro,hydro,TW0 4,2.0,0.000000,6.0,0.000000,0.900000,True,1941,100.0,0.0,,,0.0,False,0.0,inf,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False,NaN,True,0.0,0.0,0.0,2.0


In [82]:
prepared = networks.get("prepared", n)
link_cols = [c for c in ["bus0", "bus1", "carrier", "p_nom", "p_nom_min", "p_nom_max", "p_nom_extendable", "efficiency", "capital_cost"] if c in prepared.links.columns]
store_cols = [c for c in ["bus", "carrier", "e_nom", "e_nom_min", "e_nom_max", "e_nom_extendable", "capital_cost"] if c in prepared.stores.columns]
display(prepared.links[link_cols].reset_index(names="link"))
display(prepared.stores[store_cols].reset_index(names="store"))

attribute,link,bus0,bus1,carrier,p_nom,p_nom_min,p_nom_max,p_nom_extendable,efficiency,capital_cost


attribute,store,bus,carrier,e_nom,e_nom_min,e_nom_max,e_nom_extendable,capital_cost


## Renewable energy potential profiles

These NetCDF files contain the spatially clustered availability time series and capacity limits used for renewable generators.

In [83]:
profile_dir = RESOURCES / "renewable_profiles"
profile_paths = sorted(profile_dir.glob("profile_*.nc"))
profile_rows = []
profile_datasets = {}
for path in profile_paths:
    carrier = path.stem.replace("profile_", "")
    ds = xr.open_dataset(path)
    profile_datasets[carrier] = ds
    row = {
        "carrier": carrier,
        "path": str(path.relative_to(REPO)),
        "dims": dict(ds.sizes),
        "variables": ", ".join(ds.data_vars),
    }
    for var in ["profile", "p_nom_max", "potential", "average_distance", "underwater_fraction"]:
        if var in ds:
            arr = ds[var]
            row[f"{var}_mean"] = float(arr.mean(skipna=True))
            row[f"{var}_max"] = float(arr.max(skipna=True))
    profile_rows.append(row)

pd.DataFrame(profile_rows)

,carrier,path,dims,variables,profile_mean,profile_max,p_nom_max_mean,p_nom_max_max,potential_mean,potential_max,average_distance_mean,average_distance_max,underwater_fraction_mean,underwater_fraction_max
0,hydro,resources\tw_test1_gurobi_2013_7d_4h_6b\renewa...,"{'plant': 16, 'time': 144}",inflow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,offwind-ac,resources\tw_test1_gurobi_2013_7d_4h_6b\renewa...,"{'time': 144, 'bus': 66, 'y': 34, 'x': 33}","profile, weight, p_nom_max, potential, average...",0.397008,1.000000,187.510697,1722.967860,11.030041,1222.633215,20.701163,35.820747,0.544379,0.944946
2,offwind-dc,resources\tw_test1_gurobi_2013_7d_4h_6b\renewa...,"{'time': 144, 'bus': 66, 'y': 34, 'x': 33}","profile, weight, p_nom_max, potential, average...",0.190669,1.000000,574.779002,17115.006833,33.810530,3076.845650,25.029820,437.632801,0.290002,0.992194
3,onwind,resources\tw_test1_gurobi_2013_7d_4h_6b\renewa...,"{'time': 144, 'bus': 124, 'y': 14, 'x': 14}","profile, weight, p_nom_max, potential, average...",0.222541,1.000000,342.350571,2266.515551,216.589137,1903.888078,17.340082,94.581569,NaN,NaN
4,solar,resources\tw_test1_gurobi_2013_7d_4h_6b\renewa...,"{'time': 144, 'bus': 124, 'y': 14, 'x': 14}","profile, weight, p_nom_max, potential, average...",0.125114,0.686719,318.074497,1239.727200,201.230804,3062.017470,17.798552,85.481232,NaN,NaN


In [84]:
for carrier, ds in profile_datasets.items():
    print("\n==", carrier)
    display(ds)


== hydro


<xarray.Dataset> Size: 20kB
Dimensions:  (plant: 16, time: 144)
Coordinates:
  * plant    (plant) int64 128B 1 24 25 32 33 36 39 40 41 57 58 59 60 91 110 117
  * time     (time) datetime64[ns] 1kB 2013-03-01 ... 2013-03-06T23:00:00
Data variables:
    inflow   (plant, time) float64 18kB ...


== offwind-ac


<xarray.Dataset> Size: 90kB
Dimensions:              (time: 144, bus: 66, y: 34, x: 33)
Coordinates:
  * time                 (time) datetime64[ns] 1kB 2013-03-01 ... 2013-03-06T...
  * bus                  (bus) <U3 792B '1' '2' '3' '11' ... '184' '188' '192'
  * y                    (y) float64 272B 17.1 17.4 17.7 18.0 ... 26.4 26.7 27.0
  * x                    (x) float64 264B 114.0 114.3 114.6 ... 123.3 123.6
Data variables:
    profile              (time, bus) float64 76kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    weight               (bus) float64 528B ...
    p_nom_max            (bus) float64 528B 509.5 4.098 182.5 ... 42.96 98.25
    potential            (y, x) float64 9kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    average_distance     (bus) float64 528B 28.01 24.7 24.22 ... 21.89 35.21
    underwater_fraction  (bus) float64 528B 0.4992 0.4155 ... 0.8846 0.5816


== offwind-dc


<xarray.Dataset> Size: 90kB
Dimensions:              (time: 144, bus: 66, y: 34, x: 33)
Coordinates:
  * time                 (time) datetime64[ns] 1kB 2013-03-01 ... 2013-03-06T...
  * bus                  (bus) <U3 792B '1' '2' '3' '11' ... '184' '188' '192'
  * y                    (y) float64 272B 17.1 17.4 17.7 18.0 ... 26.4 26.7 27.0
  * x                    (x) float64 264B 114.0 114.3 114.6 ... 123.3 123.6
Data variables:
    profile              (time, bus) float64 76kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    weight               (bus) float64 528B ...
    p_nom_max            (bus) float64 528B 62.19 0.0 0.0 0.0 ... 41.81 0.0 0.0
    potential            (y, x) float64 9kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    average_distance     (bus) float64 528B 49.03 0.0 0.0 0.0 ... 62.53 0.0 0.0
    underwater_fraction  (bus) float64 528B 0.6903 0.01332 ... 0.01259 0.01402


== onwind


<xarray.Dataset> Size: 150kB
Dimensions:           (time: 144, bus: 124, y: 14, x: 14)
Coordinates:
  * time              (time) datetime64[ns] 1kB 2013-03-01 ... 2013-03-06T23:...
  * bus               (bus) <U3 1kB '0' '1' '4' '6' ... '189' '190' '191' '192'
  * y                 (y) float64 112B 21.6 21.9 22.2 22.5 ... 24.9 25.2 25.5
  * x                 (x) float64 112B 118.2 118.5 118.8 ... 121.5 121.8 122.1
Data variables:
    profile           (time, bus) float64 143kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    weight            (bus) float64 992B ...
    p_nom_max         (bus) float64 992B 2.646 16.66 204.3 ... 626.1 1.25e+03
    potential         (y, x) float64 2kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    average_distance  (bus) float64 992B 11.2 14.2 25.02 ... 16.8 15.75 26.55


== solar


<xarray.Dataset> Size: 150kB
Dimensions:           (time: 144, bus: 124, y: 14, x: 14)
Coordinates:
  * time              (time) datetime64[ns] 1kB 2013-03-01 ... 2013-03-06T23:...
  * bus               (bus) <U3 1kB '0' '1' '4' '6' ... '189' '190' '191' '192'
  * y                 (y) float64 112B 21.6 21.9 22.2 22.5 ... 24.9 25.2 25.5
  * x                 (x) float64 112B 118.2 118.5 118.8 ... 121.5 121.8 122.1
Data variables:
    profile           (time, bus) float64 143kB 0.1556 0.141 ... 0.02652 0.0
    weight            (bus) float64 992B ...
    p_nom_max         (bus) float64 992B 456.0 1.24e+03 154.7 ... 161.6 193.7
    potential         (y, x) float64 2kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    average_distance  (bus) float64 992B 13.36 18.28 15.34 ... 17.28 15.33 19.0

In [85]:
profile_means = []
for carrier, ds in profile_datasets.items():
    if "profile" not in ds:
        continue
    profile = ds["profile"].to_pandas()
    if isinstance(profile, pd.Series):
        profile = profile.to_frame(carrier)
    profile_means.append(profile.mean(axis=1).rename(carrier))

if profile_means:
    profile_plot_df = pd.concat(profile_means, axis=1)
    fig = px.line(profile_plot_df, title="Average renewable availability profiles")
    fig.update_yaxes(title="per-unit availability")
    fig.show()
else:
    print("No renewable profile variable named 'profile' found.")

## Bus maps and clustering trace

In [86]:
busmap_s = read_table(RESOURCES / "bus_regions" / "busmap_elec_s.csv")
busmap_s_6 = read_table(RESOURCES / "bus_regions" / "busmap_elec_s_6.csv")
display(show_head(busmap_s, 20))
display(show_head(busmap_s_6, 20))

,Unnamed: 0,0
0,1,3
1,2,3
2,4,5
3,6,8
4,7,8
5,9,11
6,10,11
7,12,13
8,14,16
9,15,16


,Bus,busmap
0,100,TW0 4
1,103,TW0 4
2,105,TW0 2
3,106,TW0 1
4,109,TW0 2
5,11,TW0 0
6,112,TW0 4
7,113,TW0 1
8,115,TW0 2
9,116,TW0 2


In [87]:
if not busmap_s_6.empty:
    cluster_counts = busmap_s_6.groupby("busmap").size().rename("precluster_bus_count").reset_index()
    display(cluster_counts)

    for cluster in ["TW1 0", "TW2 0"]:
        members = busmap_s_6.loc[busmap_s_6["busmap"].eq(cluster), "Bus"].astype(str).tolist()
        print(cluster, "members in elec_s:", members)

,busmap,precluster_bus_count
0,TW0 0,12
1,TW0 1,13
2,TW0 2,16
3,TW0 3,4
4,TW0 4,15
5,TW0 5,19


TW1 0 members in elec_s: []
TW2 0 members in elec_s: []
